In [1]:
import torch
from utils import load_splits, tokenize_dataset, compute_metrics, init_model
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import Dataset

/home/default/projects/fake_news_classification/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#global random seed
SEED = 42
torch.manual_seed(SEED)

In [3]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

True
NVIDIA GeForce RTX 3050 6GB Laptop GPU


device(type='cuda')

In [6]:
X_train, X_val, X_test, y_train, y_val, y_test = load_splits()

In [7]:
from transformers import (
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    set_seed,
)

import torch
import gc
from pathlib import Path


def training_loop(ds_train, ds_val, learning_rate, model_name, per_device_train_batch_size,
                  gradient_accumulation_step, seed=SEED):
    print(f"Training with learning rate: {learning_rate}")
    model = trainer = None
    training_args = TrainingArguments(
        output_dir=f"models/{model_name}_{learning_rate}",
        learning_rate=learning_rate,
        per_device_train_batch_size=per_device_train_batch_size,
        per_device_eval_batch_size=32,
        gradient_accumulation_steps=gradient_accumulation_step,
        num_train_epochs=5,
        weight_decay=0.01,
        fp16=torch.cuda.is_available(),
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        save_total_limit=5,
        logging_strategy="epoch",
        report_to="none",
        seed=seed,
        data_seed=seed,
    )
    try:
        # Seed Python, NumPy, and Torch before the randomly initialized classifier head.
        set_seed(seed)
        model = init_model(model_name)
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=ds_train,
            eval_dataset=ds_val,
            processing_class=tokenizer,
            data_collator=DataCollatorWithPadding(tokenizer),
            compute_metrics=compute_metrics,
        )
        trainer.train()
        # Persist the final TrainerState, including the full history and training summary.
        trainer.save_state()
        return Path(training_args.output_dir)
    finally:
        trainer = model = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


In [ ]:
model = "distilbert-base-uncased"

per_device_train_batch_size=32

gradient_accumulation_step=1

for learning_rate in [1e-5, 2e-5, 3e-5]:
    ds_train = tokenize_dataset(X_train, y_train, model)
    ds_val =  tokenize_dataset(X_val, y_val, model)     
    training_loop(ds_train, ds_val, learning_rate,
                model, per_device_train_batch_size, gradient_accumulation_step, seed=SEED)



Map:  23%|██▎       | 10000/42917 [00:00<00:01, 27211.44 examples/s]

In [ ]:
import json
from pathlib import Path

model_names = ["bert-base-uncased", "distilbert-base-uncased"]
learning_rates = [1e-7, 1e-6, 5e-6, 1e-5, 2e-5, 3e-5]
Path("logs").mkdir(exist_ok=True)

for model_name in model_names:
    for learning_rate in learning_rates:
        state_path = Path("models") / f"{model_name}_{learning_rate}" / "trainer_state.json"
        if not state_path.is_file():
            continue
        history = json.loads(state_path.read_text())["log_history"]
        training_by_epoch = {
            entry["epoch"]: entry for entry in history if "loss" in entry
        }
        results = []
        for entry in history:
            if "eval_f1_macro" in entry:
                results.append({
                    **training_by_epoch.get(entry["epoch"], {}),
                    **entry,
                    "learning_rate": learning_rate,
                })
        log_path = Path("logs") / f"{model_name}_{learning_rate}.json"
        log_path.write_text(json.dumps(results, indent=4))
        print(f"Wrote {log_path}")
